# Clinical NLP Ensemble for Pathology Reports

**UBC NLP learning project — LVI extraction**

This notebook compares a rule-based baseline with a classical TF-IDF + Logistic Regression model, demonstrates a simple rule/ML decision strategy, evaluates performance, and flags predictions for human review.

**Scope:** The dataset target used here is `lvi_label`. Nottingham grade is not modeled because a corresponding labelled target column has not been confirmed. This is an educational prototype, not a clinical decision-support system.

## 1. Load the dataset from Google Drive

The CSV is kept outside GitHub. The cell searches the expected course folder for `synthetic_breast_cancer_reports_600.csv`. Update `SEARCH_ROOT` if your folder is different.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [1]:
# Cell purpose: Purpose: perform the operation described in the section above.

from pathlib import Path
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount("/content/drive")

SEARCH_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "UBC-Extraction-and-Analysis-of-Clinical-Documents"
)
FILE_NAME = "synthetic_breast_cancer_reports_600.csv"

matches = list(SEARCH_ROOT.rglob(FILE_NAME))
if not matches:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME} under {SEARCH_ROOT}. "
        "Update SEARCH_ROOT to the folder containing your dataset."
    )

FILE_PATH = matches[0]
print(f"Dataset found: {FILE_PATH}")
df = pd.read_csv(FILE_PATH)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Mounted at /content/drive
Dataset found: /content/drive/MyDrive/Colab Notebooks/UBC-Extraction-and-Analysis-of-Clinical-Documents/synthetic_breast_cancer_reports_600.csv
Rows: 600
Columns: ['Unnamed: 0', 'report', 'lvi_label']


## 2. Validate and prepare the data

Expected columns are `report` and `lvi_label`, where `1` means LVI present and `0` means LVI absent. Any exported index column is dropped. This cell does not display report text to reduce accidental exposure of row-level clinical text in notebook outputs.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [2]:
# Cell purpose: Purpose: perform the operation described in the section above.

required = {"report", "lvi_label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}. "
        f"Found columns: {df.columns.tolist()}"
    )

df = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()
df["report"] = df["report"].fillna("").astype(str)
df["lvi_label"] = pd.to_numeric(df["lvi_label"], errors="coerce")
df = df.dropna(subset=["lvi_label"]).copy()
df["lvi_label"] = df["lvi_label"].astype(int)

if not set(df["lvi_label"].unique()).issubset({0, 1}):
    raise ValueError("Expected binary lvi_label values 0 and 1.")

df = df[df["report"].str.strip().ne("")].copy()
print("Usable rows:", len(df))
print("Empty reports removed: rows with blank report text were excluded.")
print("Label counts (0=absent, 1=present):")
display(df["lvi_label"].value_counts().sort_index())

Usable rows: 600
Empty reports removed: rows with blank report text were excluded.
Label counts (0=absent, 1=present):


,count
lvi_label,
0,322
1,278


## 3. Train/test split

The test set is held out from model fitting. For a real clinical dataset, use a patient-level split when multiple reports can belong to the same patient; this dataset's patient grouping has not been verified here.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [3]:
# Cell purpose: Purpose: perform the operation described in the section above.

from sklearn.model_selection import train_test_split

X = df["report"]
y = df["lvi_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 480
Test rows: 120


## 4. Model A — rule-based baseline

The rule-based baseline searches for common positive and negative phrases. It abstains when it finds both polarities or neither. These patterns are illustrative and are not a complete clinical terminology system.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [4]:
# Cell purpose: Purpose: perform the operation described in the section above.

import re

NEGATIVE_PATTERNS = [
    r"no lymphovascular invasion",
    r"lymphovascular invasion is not identified",
    r"lymphovascular invasion not identified",
    r"negative for lymphovascular invasion",
    r"no evidence of lymphovascular invasion",
    r"lvi is absent",
    r"lvi negative",
]
POSITIVE_PATTERNS = [
    r"lymphovascular invasion is present",
    r"lymphovascular invasion present",
    r"positive for lymphovascular invasion",
    r"identified lymphovascular invasion",
    r"lvi is present",
    r"lvi positive",
]

def rule_based_predict(text):
    text = str(text).lower()
    neg = any(re.search(pattern, text) for pattern in NEGATIVE_PATTERNS)
    pos = any(re.search(pattern, text) for pattern in POSITIVE_PATTERNS)
    if neg and not pos:
        return 0
    if pos and not neg:
        return 1
    return np.nan

rule_test = X_test.apply(rule_based_predict)
print(f"Rule-based coverage: {rule_test.notna().mean():.1%}")
print(f"Abstentions/uncertain: {int(rule_test.isna().sum())} of {len(rule_test)}")

Rule-based coverage: 13.3%
Abstentions/uncertain: 104 of 120


## 5. Model B — classical machine learning

TF-IDF converts text into numerical features; Logistic Regression learns from the training reports. The model is fitted only on the training split.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [5]:
# Cell purpose: Purpose: perform the operation described in the section above.

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

ml_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000)),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    )),
])

ml_model.fit(X_train, y_train)
ml_pred = ml_model.predict(X_test)
ml_prob = ml_model.predict_proba(X_test)[:, 1]
print("Classical ML model trained.")

Classical ML model trained.


## 6. Combine predictions

This simple decision strategy uses the rule-based result when it agrees with the ML result. If the rule-based model abstains or the two models disagree, the ML prediction is used as a fallback. Because this fallback is not a true independent majority vote, it is labelled as a **rule/ML fallback ensemble**.

The lecture's probability-averaging example is shown separately. The rule-based model does not produce calibrated probabilities, so its output is not included in probability averaging.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [6]:
# Cell purpose: Purpose: perform the operation described in the section above.

ensemble_pred = []
for rule_value, ml_value in zip(rule_test.to_numpy(), ml_pred):
    if pd.isna(rule_value):
        ensemble_pred.append(int(ml_value))
    elif int(rule_value) == int(ml_value):
        ensemble_pred.append(int(rule_value))
    else:
        ensemble_pred.append(int(ml_value))  # documented ML fallback

ensemble_pred = np.asarray(ensemble_pred, dtype=int)

lecture_probabilities = [0.45, 0.45, 0.90]
lecture_average = float(np.mean(lecture_probabilities))
print("Lecture probabilities:", lecture_probabilities)
print("Average probability:", round(lecture_average, 2))
print("Class at threshold 0.5:", "LVI present" if lecture_average >= 0.5 else "LVI absent")

Lecture probabilities: [0.45, 0.45, 0.9]
Average probability: 0.6
Class at threshold 0.5: LVI present


## 7. Flag predictions for human review

Flag a test case if the rule-based model abstains, the rule and ML models disagree, or the ML score is close to 0.5. This is an educational review heuristic; thresholds for clinical use require expert validation.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [7]:
# Cell purpose: Purpose: perform the operation described in the section above.

review_flags, review_reasons = [], []
for rule_value, ml_value, probability in zip(rule_test.to_numpy(), ml_pred, ml_prob):
    reasons = []
    if pd.isna(rule_value):
        reasons.append("Rule-based model uncertain/conflicting")
    elif int(rule_value) != int(ml_value):
        reasons.append("Rule and ML disagree")
    if abs(float(probability) - 0.5) <= 0.10:
        reasons.append("ML score near threshold")
    review_flags.append(bool(reasons))
    review_reasons.append("; ".join(reasons))

results = pd.DataFrame({
    "test_row_number": np.arange(len(X_test)),
    "actual_lvi_label": y_test.reset_index(drop=True),
    "rule_prediction": rule_test.reset_index(drop=True),
    "ml_prediction": ml_pred,
    "ml_score_lvi_present": ml_prob,
    "ensemble_prediction": ensemble_pred,
    "human_review": review_flags,
    "review_reason": review_reasons,
})

print(f"Flagged for review: {sum(review_flags)} of {len(results)} test cases")
display(results[["test_row_number", "actual_lvi_label", "rule_prediction",
                 "ml_prediction", "ensemble_prediction", "human_review",
                 "review_reason"]].head(15))

Flagged for review: 107 of 120 test cases


,test_row_number,actual_lvi_label,rule_prediction,ml_prediction,ensemble_prediction,human_review,review_reason
0,0,0,NaN,0,0,True,Rule-based model uncertain/conflicting
1,1,1,NaN,1,1,True,Rule-based model uncertain/conflicting
2,2,1,NaN,1,1,True,Rule-based model uncertain/conflicting; ML sco...
3,3,1,NaN,1,1,True,Rule-based model uncertain/conflicting; ML sco...
4,4,0,NaN,0,0,True,Rule-based model uncertain/conflicting
5,5,0,NaN,0,0,True,Rule-based model uncertain/conflicting
6,6,1,NaN,1,1,True,Rule-based model uncertain/conflicting
7,7,1,NaN,1,1,True,Rule-based model uncertain/conflicting; ML sco...
8,8,1,1.0,1,1,False,
9,9,1,NaN,0,0,True,Rule-based model uncertain/conflicting; ML sco...


## 8. Evaluate model performance

Metrics are computed on the held-out test set. Rule-based metrics are calculated only on cases where the rule-based model returns a prediction, so its coverage must be considered alongside its scores.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [8]:
# Cell purpose: Purpose: perform the operation described in the section above.

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def metric_row(name, truth, prediction):
    return {
        "model": name,
        "accuracy": accuracy_score(truth, prediction),
        "precision": precision_score(truth, prediction, zero_division=0),
        "recall": recall_score(truth, prediction, zero_division=0),
        "f1": f1_score(truth, prediction, zero_division=0),
    }

metric_rows = [
    metric_row("Classical ML (TF-IDF + Logistic Regression)", y_test, ml_pred),
    metric_row("Rule/ML fallback ensemble", y_test, ensemble_pred),
]

covered = rule_test.notna()
if covered.any():
    metric_rows.append(
        metric_row(
            "Rule-based (covered cases only)",
            y_test[covered],
            rule_test[covered].astype(int)
        )
    )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

print("Classical ML confusion matrix [rows=true, columns=predicted]:")
print(confusion_matrix(y_test, ml_pred))
print("Classical ML classification report:")
print(classification_report(y_test, ml_pred, zero_division=0))

,model,accuracy,precision,recall,f1
0,Classical ML (TF-IDF + Logistic Regression),0.983,1.0,0.964,0.982
1,Rule/ML fallback ensemble,0.983,1.0,0.964,0.982
2,Rule-based (covered cases only),1.000,1.0,1.000,1.000


Classical ML confusion matrix [rows=true, columns=predicted]:
[[64  0]
 [ 2 54]]
Classical ML classification report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        64
           1       1.00      0.96      0.98        56

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120



## 9. Export safe, non-text results

This exports aggregate metrics and a test prediction table without pathology report text. Do not export or commit row-level reports or other sensitive information to a public repository.

**Cell comment — what this code does:** Purpose: perform the operation described in the section above.

In [9]:
# Cell purpose: Purpose: perform the operation described in the section above.

OUTPUT_DIR = Path("/content/clinical_nlp_ensemble_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUTPUT_DIR / "evaluation_summary.csv"
predictions_path = OUTPUT_DIR / "test_predictions_no_report_text.csv"

metrics_df.to_csv(metrics_path, index=False)
results.to_csv(predictions_path, index=False)

print("Saved aggregate metrics:", metrics_path)
print("Saved predictions without report text:", predictions_path)

Saved aggregate metrics: /content/clinical_nlp_ensemble_results/evaluation_summary.csv
Saved predictions without report text: /content/clinical_nlp_ensemble_results/test_predictions_no_report_text.csv


## 10. Key takeaways and limitations

- Rule-based systems can miss terminology variation and contextual cues.
- Classical ML may learn useful patterns but can still mishandle negation and context.
- Combining models does not guarantee better performance; compare results empirically.
- A model score is not automatically a calibrated probability.
- Uncertain or conflicting predictions may warrant expert review.
- Transformer fine-tuning and LLM integration are not included in this cleaned notebook.
- The notebook focuses on LVI because `lvi_label` is the confirmed target column.
- Results on synthetic or course-assignment data do not establish clinical validity.

**Important:** This is an educational prototype only, not a clinical decision-support system. Do not use its predictions for patient care.